# Case study: Hubble in 1929, a law in noisy measurements

In 1929 Edwin Hubble published the distances and the recession velocities of 24
nebulae, and the proportion between the two. The measurements scatter widely: a
straight line explains 62 % of their variance. This notebook gives GP_ELITE those
24 rows. With units, it returns v = H·d with H in km/(Mpc·s), the line through
the origin that least squares give; without units, the line with an intercept.
It finds nothing beyond a straight line, and that is what the data support.

Every figure in the text was measured with `benchmarks/case_studies.py hubble`,
for the seeds 0 to 4 (output in `benchmarks/results_0.9/case_studies/hubble.txt`,
and with version 1.1.0 in `benchmarks/results_1.1/release/m3/case_hubble.txt`).

Run the cells in order with **Shift + Enter**. Each fit takes ten to twenty
seconds.

*[🇫🇷 Version française](https://colab.research.google.com/github/ariel95500-create/gp-elite/blob/main/examples/case_studies/hubble_1929.fr.ipynb)*

## 1. Install

In [ ]:
!pip install -q gp-elite

import gp_elite
print('GP_ELITE', gp_elite.__version__, 'ready.')

## 2. The data and their source

The table comes from the R package Sleuth2 (dataset `case0701`; Ramsey and
Schafer, *The Statistical Sleuth*, 2nd edition, case study 7.1), which
transcribes Hubble's (*Proceedings of the National Academy of Sciences* 15, 168,
1929): distance in megaparsecs, velocity in km/s. A second transcription, in the
R package lmreg, gives −200 km/s instead of +200 for the nebula at 0.63 Mpc; we
could not check the original table to settle it, and section 6 shows what it
changes.

In [ ]:
import os
import numpy as np
import pandas as pd

NAME = 'hubble1929.csv'
URL = ('https://raw.githubusercontent.com/ariel95500-create/gp-elite/main/'
       'examples/case_studies/data/' + NAME)
path = os.path.join('data', NAME)
df = pd.read_csv(path if os.path.exists(path) else URL, comment='#')

d = df['distance_mpc'].to_numpy(float)
v = df['velocity_kms'].to_numpy(float)
print(df.to_string(index=False))

## 3. The straight lines to beat

Before any search, the two lines least squares give: through the origin, as
Hubble's law has it, and with an intercept.

In [ ]:
def r2(p, t):
    return 1 - np.mean((p - t) ** 2) / np.var(t)

def rmse(p, t):
    return np.sqrt(np.mean((p - t) ** 2))

k0 = d @ v / (d @ d)              # through the origin
b1, b0 = np.polyfit(d, v, 1)      # with an intercept
print(f'through the origin: v = {k0:.2f} d             '
      f'R2 {r2(k0 * d, v):.4f}   RMSE {rmse(k0 * d, v):.1f} km/s')
print(f'with an intercept : v = {b0:.2f} + {b1:.2f} d   '
      f'R2 {r2(b0 + b1 * d, v):.4f}   RMSE {rmse(b0 + b1 * d, v):.1f} km/s')

Through the origin, v = 423.94·d, R² 0.6194, root mean square error 224 km/s.
With an intercept, v = −40.78 + 454.16·d, R² 0.6235, 223 km/s.

## 4. GP_ELITE, with and without units

In [ ]:
from gp_elite import GPEliteRegressor

X = d.reshape(-1, 1)

with_units = GPEliteRegressor(units=['Mpc'], target_units='km/s',
                              unknown_constant=True, generations=30,
                              speed='fast', random_state=0, parallel=False)
with_units.fit(X, v)
p = with_units.predict(X)
print('with units   :', with_units.equation_)
print('constant     :', with_units.constant_value_, with_units.constant_units_string())
print(f'               R2 {r2(p, v):.4f}   RMSE {rmse(p, v):.1f} km/s')

no_units = GPEliteRegressor(generations=30, speed='fast', random_state=0,
                            parallel=False)
no_units.fit(X, v)
p = no_units.predict(X)
print('without units:', no_units.equation_)
print(f'               R2 {r2(p, v):.4f}   RMSE {rmse(p, v):.1f} km/s')

With units, the engine returns `423.937 * X0`: the line through the origin of
least squares, and its constant in km/(Mpc·s), the units of Hubble's constant.
A velocity cannot be a bare number, so under units the line cannot take an
intercept, and passes through the origin as Hubble's law does. Without units,
it returns `-40.7836 + 454.158 * X0`, the line with an intercept, which lowers
the error a little (223.0 km/s against 224.2). The seeds 0 to 4 all return
these two formulas.

The constant, 424 km/(Mpc·s), is Hubble's constant as these data give it.
Hubble's distances were later found to be much too short, and the value
measured today is several times smaller; no fit of these 24 rows can see that.

**What the noise-tolerant seeds do here.** This version of the engine places,
when it finds no exact form, a form that explains the data to within 1.5 % of
their variance. Here the best line leaves 38 % of it: these seeds place
nothing, and neither do the exact ones, with or without units. What you see
is the plain search, which finds the line on its own.

## 5. A held-out check: the six farthest nebulae

Both fits again on the 18 nebulae up to 1.1 Mpc, predicting the 6 beyond
(1.4 to 2.0 Mpc).

In [ ]:
near, far = d <= 1.1, d > 1.1
k_near = d[near] @ v[near] / (d[near] @ d[near])
print(f'{near.sum()} nebulae to fit, {far.sum()} to predict')
print(f'least squares through the origin: v = {k_near:.2f} d   '
      f'RMSE on the 6: {rmse(k_near * d[far], v[far]):.1f} km/s')

for label, kw in (('with units', dict(units=['Mpc'], target_units='km/s',
                                      unknown_constant=True)),
                  ('without units', {})):
    m = GPEliteRegressor(generations=30, speed='fast', random_state=0,
                         parallel=False, **kw)
    m.fit(X[near], v[near])
    print(f'{label:13s}: {m.equation_}   '
          f'RMSE on the 6: {rmse(m.predict(X[far]), v[far]):.1f} km/s')

With units, `427.063 * X0`, and an error of 203 km/s on the six far nebulae, the
same as least squares through the origin. Without units, `-74.9921 + 520.269 *
X0`, and 231 km/s. Here the law through the origin carries further; on six rows
this scattered, one would not read more into it.

## 6. The other transcription

The same two fits with −200 km/s for the nebula at 0.63 Mpc.

In [ ]:
v_lmreg = v.copy()
v_lmreg[9] = -200.0     # the tenth row, 0.63 Mpc, as in the R package lmreg

for label, kw in (('with units', dict(units=['Mpc'], target_units='km/s',
                                      unknown_constant=True)),
                  ('without units', {})):
    m = GPEliteRegressor(generations=30, speed='fast', random_state=0,
                         parallel=False, **kw)
    m.fit(X, v_lmreg)
    print(f'{label:13s}: {m.equation_}   R2 {r2(m.predict(X), v_lmreg):.4f}')

With units, `415.4 * X0`: one row out of 24 moves the constant by 2 %. Without
units, `-68.1539 + 465.903 * X0`.

## 7. What the engine does not find

- **Anything beyond a straight line.** On these 24 rows the engine returns the
  line, with or without its intercept, and nothing more.
- **Why the points scatter.** Errors on the distances and the nebulae's own
  motions are not in the data.
- **The true value of H.** The engine fits the data it is given; whether the
  distances are right is a question for astronomers.

## 8. The five seeds

The seeds 0 to 4 return the same formulas, to the last printed digit, in every
fit of this notebook. The figures were measured on the development branch that follows version 0.9.0,
then again on the packages of versions 1.0.0 and 1.1.0, with the same results to
the last printed digit (`benchmarks/results_0.9/release_1.0/m3/`,
`benchmarks/results_1.1/release/m3/`).

The full output: `benchmarks/results_1.1/release/m3/case_hubble.txt` (version
1.1.0), and `benchmarks/results_0.9/case_studies/hubble.txt` for the first
measurement. The protocol,
written before the measurements: `benchmarks/results_0.9/PLAN_PHASE6.md`.